In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, TimestampType,DoubleType
)

In [0]:
bronze = 'abfss://bronze@adlscshm.dfs.core.windows.net/instacart/'
volume = '/Volumes/proyecto/raw/raw_data/data_proyecto/'
tables = ['orders', 'order_products', 'products', 'aisles', 'departments']

In [0]:
for i in tables:
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS proyecto.bronze.{i}
    USING DELTA
    LOCATION '{bronze}{i}'
    """)


In [0]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("eval_set", StringType(), True),
    StructField("order_number", IntegerType(), True),
    StructField("order_dow", IntegerType(), True),
    StructField("order_hour_of_day", IntegerType(), True),
    StructField("days_since_prior_order", DoubleType(), True),

])

order_products_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("add_to_cart_order", IntegerType(), True),
    StructField("reordered", IntegerType(), True),

])

products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("aisle_id", IntegerType(), True),
    StructField("department_id", IntegerType(), True),

])

aisles_schema = StructType([
    StructField("aisle_id", IntegerType(), True),
    StructField("aisle", StringType(), True),

])

departments_schema = StructType([
    StructField("department_id", IntegerType(), True),
    StructField("department", StringType(), True),

])

In [0]:
catalog = "proyecto"
schema = "bronze"

orders_data = f"{volume}orders.csv"
order_products_data = f"{volume}order_products.csv"
products_data = f"{volume}products.csv"
aisles_data = f"{volume}aisles.csv"
departments_data = f"{volume}departments.csv"

In [0]:
# Orders
df_orders = spark.read.schema(orders_schema).csv(orders_data, header=True, sep=",")
df_orders.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{catalog}.{schema}.orders")

# Order Products
df_order_products = spark.read.schema(order_products_schema).csv(order_products_data, header=True, sep=",")
df_order_products.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.order_products")

# Products
df_products = spark.read.schema(products_schema).csv(products_data, header=True, sep=",")
df_products.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.products")

# Aisles
df_aisles = spark.read.schema(aisles_schema).csv(aisles_data, header=True, sep=",")
df_aisles.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.aisles")

# Departments
df_departments = spark.read.schema(departments_schema).csv(departments_data, header=True, sep=",")
df_departments.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.departments")


In [0]:
%sql
select * 
from proyecto.bronze.orders